### Pandapower with UK Power Networks - sensitivity-based grid reduction

This tutorial complements the tutorial presented [here](https://github.com/e2nIEE/pandapower/blob/develop/tutorials/ukpn_pp_power_flow.ipynb), which shows how to leverage pandapower for performing studies and analyses on the real grids of UK Power Networks. 
This tutorial shows how to create reduced models of the grids of UK Power Networks to perform simplified analyses and investigations on smaller sections.

This tutorial has been created in collaboration with UK Power Networks, the Distribution System Operator owning and operating the electricity network across London, the South East and the East of England.

The tutorial will use the real grids associated with the three licensed electricity distribution networks operated by UK Power Networks (LPN, SPN and EPN).
It will present the functionalities created to reduce the grid and the different settings available to customize the grid reduction as desired..

UK Power Networks has provided the grid data as part of their LTDS CIM dataset release. It is a "Shared" dataset that requires special access. To request access, visit the [LTDS CIM](https://ukpowernetworks.opendatasoft.com/explore/dataset/ukpn-ltds-cim/information/) page and complete the [Shared Data Request Form](https://ukpowernetworks.opendatasoft.com/login/?next=/explore/forms/cim-access-request-form/). Once approved, CIM data is published as XML file attachments (one per licence area: EPN, SPN, LPN). You can download the XML files directly from the portal.

The additional data required to integrate load and generation in the grid are openly available as Excel tables at the following links: 
- EPN --> [EPN Long Term Development Statement - November 2025](https://ukpowernetworks.sharepoint.com/sites/OpenDataPortalLibrary/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FOpenDataPortalLibrary%2FShared%20Documents%2FGeneral%2FLong%20Term%20Development%20Statement%2FNovember%202025%2FEPN%20Long%20Term%20Development%20Statement%20%2D%20November%202025&p=true&ga=1)
- SPN --> [SPN Long Term Development Statement - November 2025](https://ukpowernetworks.sharepoint.com/sites/OpenDataPortalLibrary/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FOpenDataPortalLibrary%2FShared%20Documents%2FGeneral%2FLong%20Term%20Development%20Statement%2FNovember%202025%2FSPN%20Long%20Term%20Development%20Statement%20%2D%20November%202025&p=true&ga=1)
- LPN --> [LPN Long Term Development Statement - November 2025](https://ukpowernetworks.sharepoint.com/sites/OpenDataPortalLibrary/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FOpenDataPortalLibrary%2FShared%20Documents%2FGeneral%2FLong%20Term%20Development%20Statement%2FNovember%202025%2FLPN%20Long%20Term%20Development%20Statement%20%2D%20November%202025&p=true&ga=1)


In [ ]:
# Import the needed libraries 
import pandapower as pp
import pandapower.topology as top
from pandapower.toolbox import create_replacement_switch_for_branch, select_subnet

import pandas as pd
import numpy as np
import copy
import os
pd.options.display.float_format = '{:,.4f}'.format

import warnings
warnings.filterwarnings('ignore')

#### Import of the UK Power Network grids
This tutorial assumes that the grids of UK Power Networks have been already imported from the CIM data and saved as pandapower networks in json format. 
To see how to import the UK Power Networks grids starting from the CIM files downloadable from the UK Power Networks portal, please refer to the following [UKPN_CIM2pp_tutorial](). 
Here you can also find how to save the pandapower grid into a json file and how to navigate through the pandapower grid data or the attributes of the different grid components. 

In [ ]:
# Import the grid for the analysis
filename = "LPN EQ SSH_0401_eq.json"   # Give here the name of the json file with the UKPN grid you want to use
if os.path.isfile(filename):
    net = pp.from_json(filename)
else:
    print("file does not exist, creating a dummy net")
    net = pp.create_empty_network()
    bus = pp.create_bus(net, vn_kv=132)
    pp.create_ext_grid(net, bus=bus)

#### Workarounds for power flow execution
The following blocks of code provide some functions to apply some workarounds necessary to run successfully the power flow on the UK Power Networks grids.
These workarounds include, for example, the creation of external grids (*slack buses* in the power flow terminology) or the replacement of zero impedance components with switches. 

In [ ]:
# Function to replace components with very small impedance with switches.
from pandapower.toolbox import create_replacement_switch_for_branch

def _replace_zero_impedance_components(net):
    min_ohm = 0.001
    to_replace = (np.abs(net.line.x_ohm_per_km * net.line.length_km) <= min_ohm) & net.line.in_service

    if np.any(to_replace):
        print(f"replaced {sum(to_replace)} lines with switches")

    for i in net.line.loc[to_replace].index.values:
        create_replacement_switch_for_branch(net, "line", i)
        net.line.at[i, "in_service"] = False

    xward = net.xward.loc[(np.abs(net.xward.x_ohm) <= min_ohm) & net.xward.in_service].index.values
    if len(xward) > 0:
        pp.replace_xward_by_ward(net, index=xward, drop=False)
        print(f"replaced {len(xward)} xwards with wards")

    zb_f_ohm = np.square(net.bus.loc[net.impedance.from_bus.values, "vn_kv"].values) / net.impedance.sn_mva
    zb_t_ohm = np.square(net.bus.loc[net.impedance.to_bus.values, "vn_kv"].values) / net.impedance.sn_mva
    impedance = ((np.abs(net.impedance.xft_pu) <= min_ohm / zb_f_ohm) |
                (np.abs(net.impedance.xtf_pu) <= min_ohm / zb_t_ohm)) & net.impedance.in_service

    if any(impedance):
        print(f"replaced {sum(impedance)} impedance elements with switches")

    for i in net.impedance.loc[impedance].index.values:
        pp.create_replacement_switch_for_branch(net, "impedance", i)
        net.impedance.at[i, "in_service"] = False

In [ ]:
# Function to apply the needed workarounds
def apply_workarounds(net, license_area, remove_impedance):
    if remove_impedance:
        net.impedance.drop(net.impedance.index, inplace=True)
    _replace_zero_impedance_components(net)
    net.line["c_nf_per_km"] *= 0.1
    net.load["p_mw"] *= 0.1

    if license_area == "LPN":
        pp.create_ext_grid(net,bus=10711,vm_pu=1)
        pp.create_ext_grid(net,bus=10699,vm_pu=1)
        pp.create_ext_grid(net,bus=10674,vm_pu=1)
        pp.create_ext_grid(net,bus=10738,vm_pu=1)
        pp.create_ext_grid(net,bus=10673,vm_pu=1)
    elif license_area == "SPN":
        net.trafo.drop(661,inplace=True)
        pp.create_ext_grid(net,bus=4899,vm_pu=1)
        pp.create_ext_grid(net,bus=4879,vm_pu=1)
        pp.create_ext_grid(net,bus=4903,vm_pu=1)
        pp.create_ext_grid(net,bus=4920,vm_pu=1)
        pp.create_ext_grid(net,bus=4916,vm_pu=1)
        pp.create_ext_grid(net,bus=4925,vm_pu=1)
        pp.create_ext_grid(net,bus=4878,vm_pu=1)
    elif license_area == "EPN":
        pp.create_ext_grid(net,bus=9906,vm_pu=1)
        pp.create_ext_grid(net,bus=9918,vm_pu=1)
        pp.create_ext_grid(net,bus=9900,vm_pu=1)
        pp.create_ext_grid(net,bus=9910,vm_pu=1)
        pp.create_ext_grid(net,bus=9878,vm_pu=1)
    else:
        raise ValueError("Sorry, this license area does not exist in UK Power Networks. Allowed areas are LPN, SPN and EPN.")

    return net

In [ ]:
# Apply the workarounds on the selected grid
license_area = "LPN"  # Provide here the name of the considered license area. It should be "LPN", "SPN", or "EPN".
if net.bus.index.size > 1:
    remove_impedance = True     # Decide if removing fictious impedances from the grid or not
    net = apply_workarounds(net, license_area, remove_impedance)

#### Sensitivity-based grid reduction
The following blocks implement the functions necessary to carry out the desired grid reduction based on sensitivity factors. 

The **goal** of the grid reduction is to reduce the grid around a user-selected bus of interest while keeping, inside the reduced grid, the same power flow behaviour as in the original-size grid. 

The main **criterion** for the grid reduction is to cut the grid at transformer level based on the sensitivity of the transformers to the changes applied at the bus of interest. In this way, only the portion of the grid directly affected by changes at the bus of interest is kept within the reduced grid model, whereas other parts of the grid that are not influenced by power variations at the bus of interest are excluded from the model and replaced with equivalent elements. 

This grid reduction process allows therefore to create reduced grid models around a selected bus and to focus the analysis on a smaller (and hence more easily manageable) portion of the UK Power Networks grid. 


Functions to compute sensitivities:

In [ ]:
def calc_trafo_current_sensitivity_from_power_flow(net_start, net_post, min_i_ka=1e-6):
    """
    Function to compute the sensitivity of transformers to a power change at the bus of interest
    """
    rows = []
    for tidx, tr in net_start.trafo[net_start.trafo.in_service].iterrows():
        tidx = int(tidx)

        hv = int(tr.hv_bus)
        lv = int(tr.lv_bus)

        # initial currents from PF
        i0_hv_ka_start = abs(float(net_start.res_trafo.i_hv_ka.loc[tidx]))
        i0_lv_ka_start = abs(float(net_start.res_trafo.i_lv_ka.loc[tidx]))

        # currents after perturbation from PF
        i0_hv_ka_post = abs(float(net_post.res_trafo.i_hv_ka.loc[tidx]))
        i0_lv_ka_post = abs(float(net_post.res_trafo.i_lv_ka.loc[tidx]))

        # current difference between before and after perturbation
        di_hv_ka = i0_hv_ka_start - i0_hv_ka_post
        di_lv_ka = i0_lv_ka_start - i0_lv_ka_post

        # sensitivity computation
        sf_hv = di_hv_ka / max(i0_hv_ka_start, float(min_i_ka)) if np.isfinite(di_hv_ka) else np.nan
        sf_lv = di_lv_ka / max(i0_lv_ka_start, float(min_i_ka)) if np.isfinite(di_lv_ka) else np.nan

        rows.append({
            "trafo_index": tidx,
            "hv_bus": hv,
            "lv_bus": lv,
            "vn_hv_kv": float(net_start.bus.vn_kv.loc[hv]),
            "vn_lv_kv": float(net_start.bus.vn_kv.loc[lv]),
            "i0_hv_ka": i0_hv_ka_start,
            "i0_lv_ka": i0_lv_ka_start,
            "i0_max_ka": max(i0_hv_ka_start, i0_lv_ka_start),
            "dI_hv_ka": float(di_hv_ka) if np.isfinite(di_hv_ka) else np.nan,
            "dI_lv_ka": float(di_lv_ka) if np.isfinite(di_lv_ka) else np.nan,
            "dI_max_ka": max(di_hv_ka, di_lv_ka),
            "sf_hv": float(sf_hv) if np.isfinite(sf_hv) else np.nan,
            "sf_lv": float(sf_lv) if np.isfinite(sf_lv) else np.nan,
            "sf_max": float(abs(np.nanmax([sf_hv, sf_lv]))),
        })

    return pd.DataFrame(rows).set_index("trafo_index")


def calc_trafo3w_current_sensitivity_from_power_flow(net_start, net_post, min_i_ka=1e-6):
    """
    Function to compute the sensitivity of 3-winding transformers to a power change at the bus of interest
    """

    rows = []
    for tidx, tr in net_start.trafo3w[net_start.trafo3w.in_service].iterrows():
        tidx = int(tidx)

        hv = int(tr.hv_bus)
        mv = int(tr.mv_bus)
        lv = int(tr.lv_bus)

        # initial currents from PF
        i0_hv_ka_start = abs(float(net_start.res_trafo3w.i_hv_ka.loc[tidx])) if "i_hv_ka" in net_start.res_trafo3w.columns else np.nan
        i0_mv_ka_start = abs(float(net_start.res_trafo3w.i_mv_ka.loc[tidx])) if "i_mv_ka" in net_start.res_trafo3w.columns else np.nan
        i0_lv_ka_start = abs(float(net_start.res_trafo3w.i_lv_ka.loc[tidx])) if "i_lv_ka" in net_start.res_trafo3w.columns else np.nan

        # currents after perturbation from PF
        i0_hv_ka_post = abs(float(net_post.res_trafo3w.i_hv_ka.loc[tidx])) if "i_hv_ka" in net_post.res_trafo3w.columns else np.nan
        i0_mv_ka_post = abs(float(net_post.res_trafo3w.i_mv_ka.loc[tidx])) if "i_mv_ka" in net_post.res_trafo3w.columns else np.nan
        i0_lv_ka_post = abs(float(net_post.res_trafo3w.i_lv_ka.loc[tidx])) if "i_lv_ka" in net_post.res_trafo3w.columns else np.nan

        # current difference between before and after perturbation
        di_hv_ka = i0_hv_ka_start - i0_hv_ka_post
        di_mv_ka = i0_mv_ka_start - i0_mv_ka_post
        di_lv_ka = i0_lv_ka_start - i0_lv_ka_post

        # sensitivity computation
        sf_hv = di_hv_ka / max(i0_hv_ka_start, float(min_i_ka)) if np.isfinite(di_hv_ka) else np.nan
        sf_mv = di_mv_ka / max(i0_mv_ka_start, float(min_i_ka)) if np.isfinite(di_mv_ka) else np.nan
        sf_lv = di_lv_ka / max(i0_lv_ka_start, float(min_i_ka)) if np.isfinite(di_lv_ka) else np.nan

        rows.append({
            "trafo3w_index": tidx,
            "hv_bus": hv,
            "mv_bus": mv,
            "lv_bus": lv,
            "vn_hv_kv": float(net_start.bus.vn_kv.loc[hv]),
            "vn_mv_kv": float(net_start.bus.vn_kv.loc[mv]),
            "vn_lv_kv": float(net_start.bus.vn_kv.loc[lv]),
            "i0_hv_ka": float(i0_hv_ka_start) if np.isfinite(i0_hv_ka_start) else np.nan,
            "i0_mv_ka": float(i0_mv_ka_start) if np.isfinite(i0_mv_ka_start) else np.nan,
            "i0_lv_ka": float(i0_lv_ka_start) if np.isfinite(i0_lv_ka_start) else np.nan,
            "i0_max_ka": max(i0_hv_ka_start, i0_mv_ka_start, i0_lv_ka_start),
            "dI_hv_ka": float(di_hv_ka) if np.isfinite(di_hv_ka) else np.nan,
            "dI_mv_ka": float(di_mv_ka) if np.isfinite(di_mv_ka) else np.nan,
            "dI_lv_ka": float(di_lv_ka) if np.isfinite(di_lv_ka) else np.nan,
            "dI_max_ka": max(di_hv_ka, di_mv_ka, di_lv_ka),
            "sf_hv": float(abs(sf_hv)) if np.isfinite(sf_hv) else np.nan,
            "sf_mv": float(abs(sf_mv)) if np.isfinite(sf_mv) else np.nan,
            "sf_lv": float(abs(sf_lv)) if np.isfinite(sf_lv) else np.nan,
            "sf_max": float(abs(np.nanmax([sf_hv, sf_mv, sf_lv]))),
        })

    return pd.DataFrame(rows).set_index("trafo3w_index")


def calc_impedance_current_sensitivity_from_power_flow(net_start, net_post, min_i_ka=1e-6):
    """
    Function to compute the sensitivity of impedance elements to a power change at the bus of interest. 
    Only impedances connecting buses at different voltage levels are taken into account.
    """

    rows = []
    for iidx, imp in net_start.impedance[net_start.impedance.in_service].iterrows():
        iidx = int(iidx)
        fb = int(imp.from_bus)
        tb = int(imp.to_bus)

        fv = net_start.bus.vn_kv.loc[fb]
        tv = net_start.bus.vn_kv.loc[tb]

        if fv == tv:
            continue

        # initial currents from PF
        i0_from_ka_start = abs(float(net_start.res_impedance.i_from_ka.loc[iidx])) if "i_from_ka" in net_start.res_impedance.columns else np.nan
        i0_to_ka_start = abs(float(net_start.res_impedance.i_to_ka.loc[iidx])) if "i_to_ka" in net_start.res_impedance.columns else np.nan

        # currents after perturbation from PF
        i0_from_ka_post = abs(float(net_post.res_impedance.i_from_ka.loc[iidx])) if "i_from_ka" in net_post.res_impedance.columns else np.nan
        i0_to_ka_post = abs(float(net_post.res_impedance.i_to_ka.loc[iidx])) if "i_to_ka" in net_post.res_impedance.columns else np.nan

        # current difference between before and after perturbation
        di_from_ka = i0_from_ka_start - i0_from_ka_post
        di_to_ka = i0_to_ka_start - i0_to_ka_post

        # sensitivity computation
        sf_from = di_from_ka / max(i0_from_ka_start, float(min_i_ka)) if np.isfinite(di_from_ka) else np.nan
        sf_to = di_to_ka / max(i0_to_ka_start, float(min_i_ka)) if np.isfinite(di_to_ka) else np.nan

        rows.append({
            "impedance_index": iidx,
            "from_bus": fb,
            "to_bus": tb,
            "vn_from_kv": float(net_start.bus.vn_kv.loc[fb]),
            "vn_to_kv": float(net_start.bus.vn_kv.loc[tb]),
            "i0_from_ka": float(i0_from_ka_start) if np.isfinite(i0_from_ka_start) else np.nan,
            "i0_to_ka": float(i0_to_ka_start) if np.isfinite(i0_to_ka_start) else np.nan,
            "i0_max_ka": max(i0_from_ka_start, i0_to_ka_start),
            "dI_from_ka": float(di_from_ka) if np.isfinite(di_from_ka) else np.nan,
            "dI_to_ka": float(di_to_ka) if np.isfinite(di_to_ka) else np.nan,
            "dI_max_ka": max(di_from_ka, di_to_ka),
            "sf_from": float(abs(sf_from)) if np.isfinite(sf_from) else np.nan,
            "sf_to": float(abs(sf_to)) if np.isfinite(sf_to) else np.nan,
            "sf_max": float(abs(np.nanmax([sf_from, sf_to]))),
        })

    if len(rows):
        return pd.DataFrame(rows).set_index("impedance_index")


def create_sets(net):
    """
    Function to identify the set of trafo, 3w-trafo and impedance to be considered in the sensitivity analysis.
    """

    el_pairs = set()
    el_adj = {}

    # ------------------------------------------------------------------
    # 2W trafos
    # ------------------------------------------------------------------
    for tidx, tr in net.trafo[net.trafo.in_service].iterrows():
        hv = int(tr.hv_bus)
        lv = int(tr.lv_bus)

        el_pairs.add(frozenset((hv, lv)))
        el_adj.setdefault(hv, []).append((lv, ("trafo", int(tidx))))
        el_adj.setdefault(lv, []).append((hv, ("trafo", int(tidx))))

    # ------------------------------------------------------------------
    # 3W trafos
    # ------------------------------------------------------------------
    for tidx, tr in net.trafo3w[net.trafo3w.in_service].iterrows():
        hv = int(tr.hv_bus)
        mv = int(tr.mv_bus)
        lv = int(tr.lv_bus)

        # all winding pairs exist electrically
        el_pairs.add(frozenset((hv, mv)))
        el_pairs.add(frozenset((hv, lv)))
        el_pairs.add(frozenset((mv, lv)))

        el_adj.setdefault(hv, []).append((mv, ("trafo3w", int(tidx))))
        el_adj.setdefault(hv, []).append((lv, ("trafo3w", int(tidx))))
        el_adj.setdefault(mv, []).append((hv, ("trafo3w", int(tidx))))
        el_adj.setdefault(mv, []).append((lv, ("trafo3w", int(tidx))))
        el_adj.setdefault(lv, []).append((hv, ("trafo3w", int(tidx))))
        el_adj.setdefault(lv, []).append((mv, ("trafo3w", int(tidx))))

    # ------------------------------------------------------------------
    # Impedances
    # ------------------------------------------------------------------
    for iidx, imp in net.impedance[net.impedance.in_service].iterrows():
        fb = int(imp.from_bus)
        tb = int(imp.to_bus)

        fv = net.bus.vn_kv.loc[fb]
        tv = net.bus.vn_kv.loc[tb]

        if fv == tv:
            continue
        
        if fv > tv:
            el_pairs.add(frozenset((fb, tb)))
        else:
            el_pairs.add(frozenset((tb, fb)))
        el_adj.setdefault(fb, []).append((tb, ("impedance", int(iidx))))
        el_adj.setdefault(tb, []).append((fb, ("impedance", int(iidx))))

    return el_pairs, el_adj


Function to cut the grid based on: 
- sensitivity value of the element (trafo, trafo3w, impedance)
- max voltage limit
- min current limit 

In [ ]:
def cut_by_sensitivity(net, graph, start_bus, trafo_sens_df, trafo3w_sens_df=None, impedance_sens_df=None,
                    sensitivity_threshold=0.05, min_working_current_ka=1e-4, di_min_ka=1e-4, vn_max_kv=None, 
                    cut_downward_elements=True, keep_boundary_outside_bus=True):
    """
    Traversal from start_bus:
      - elements are cut if:
            sf_max < sensitivity_threshold
            OR dI_max_ka <= dI_min_ka
            OR to_vn > vn_max_kv

    Returns
    -------
    kept_buses : set[int]
    boundaries : list[dict]
    """
    start_bus = int(start_bus)
    vn = net.bus.vn_kv.astype(float)

    el_pairs, el_adj = create_sets(net)

    kept_buses = {start_bus}
    visited = {start_bus}
    queue = [start_bus]
    upper_boundaries = []
    lower_boundaries = []
    visited_element_dir = set()

    while queue:
        u = queue.pop(0)

        # --------------------------------------------------------------
        # 1) non-trafo (or impedance) neighbors
        # --------------------------------------------------------------
        for v in graph.neighbors(u):
            v = int(v)
            if (frozenset((u, v)) in el_pairs) or (frozenset((v, u)) in el_pairs):
                continue
            if v not in visited:
                visited.add(v)
                kept_buses.add(v)
                queue.append(v)

        # --------------------------------------------------------------
        # 2) trafo or impedance neighbors
        # --------------------------------------------------------------
        for v, el_id in el_adj.get(u, []):
            v = int(v)
            if v > u:
                key = (el_id, int(u), int(v))
            else:
                key = (el_id, int(v), int(u))
            if key in visited_element_dir:
                continue
            visited_element_dir.add(key)

            vn_u = float(vn.loc[u])
            vn_v = float(vn.loc[v])

            # ==========================================================
            # 2W TRAFO
            # ==========================================================
            if el_id[0] == "trafo":
                tidx = int(el_id[1])

                # local upward traversal?
                is_upward = vn_v > vn_u + 1e-9

                if is_upward or cut_downward_elements:

                    # sensitivity data
                    if tidx in trafo_sens_df.index:
                        sf = float(trafo_sens_df.at[tidx, "sf_max"]) if "sf_max" in trafo_sens_df.columns else np.nan
                        i0 = float(trafo_sens_df.at[tidx, "i0_max_ka"]) if "i0_max_ka" in trafo_sens_df.columns else np.nan
                        di = float(trafo_sens_df.at[tidx, "dI_max_ka"]) if "dI_max_ka" in trafo_sens_df.columns else np.nan
                    else:
                        sf = np.nan
                        i0 = np.nan
                        di = np.nan

                    cut_due_to_vn = (
                        vn_max_kv is not None 
                        and vn_v > float(vn_max_kv) + 1e-9
                    )

                    cut_due_to_sens = (
                        np.isfinite(sf)
                        and np.isfinite(i0)
                        and i0 >= float(min_working_current_ka)
                        and sf < float(sensitivity_threshold)
                    )

                    cut_due_to_di_min = (
                        np.isfinite(di)
                        and abs(di) < float(di_min_ka)
                    )

                    if cut_due_to_vn or cut_due_to_sens or cut_due_to_di_min:
                        tr = net.trafo.loc[tidx]
                        trafo_info = {
                            "el_type": "trafo",
                            "el_index": tidx,
                            "hv_bus": int(tr.hv_bus),
                            "lv_bus": int(tr.lv_bus),
                            "boundary_bus_inside": int(u),
                            "boundary_bus_outside": int(v),
                            "reason": "vn_above_vn_max" if cut_due_to_vn else "up_below_current_sensitivity",
                            "from_vn_kv": vn_u,
                            "to_vn_kv": vn_v,
                            "sf_max": sf,
                            "i0_max_ka": i0,
                            "dI_max_ka": di,
                            "sensitivity_threshold": float(sensitivity_threshold),
                            "min_working_current_ka": float(min_working_current_ka),
                            "dI_min_ka": float(di_min_ka),
                        }
                        if is_upward:
                            upper_boundaries.append(trafo_info)
                        else:
                            lower_boundaries.append(trafo_info)

                        if keep_boundary_outside_bus:
                            kept_buses.add(int(v))
                        continue

                if v not in visited:
                    visited.add(v)
                    kept_buses.add(v)
                    queue.append(v)

                continue

            # ==========================================================
            # 3W TRAFO
            # ==========================================================
            if el_id[0] == "trafo3w":
                tidx = int(el_id[1])
                tr3 = net.trafo3w.loc[tidx]

                hv = int(tr3.hv_bus)
                mv = int(tr3.mv_bus)
                lv = int(tr3.lv_bus)

                if hv not in {u, v}: 
                    z = hv
                elif mv not in {u, v}:
                    z = mv
                else:
                    z = lv
                vn_z = float(vn.loc[z])

                if z > u:
                    key = (el_id, int(u), int(z))
                else:
                    key = (el_id, int(z), int(u))
                visited_element_dir.add(key)

                if v > z:
                    key = (el_id, int(z), int(v))
                else:
                    key = (el_id, int(v), int(z))
                visited_element_dir.add(key)

                is_upward = (vn_v > vn_u + 1e-9) or (vn_z > vn_u + 1e-9)

                if is_upward or cut_downward_elements:

                    if trafo3w_sens_df is not None and tidx in trafo3w_sens_df.index:
                        sf = float(trafo3w_sens_df.at[tidx, "sf_max"]) if "sf_max" in trafo3w_sens_df.columns else np.nan
                        i0 = float(trafo3w_sens_df.at[tidx, "i0_max_ka"]) if "i0_max_ka" in trafo3w_sens_df.columns else np.nan
                        di = float(trafo3w_sens_df.at[tidx, "dI_max_ka"]) if "dI_max_ka" in trafo3w_sens_df.columns else np.nan
                    else:
                        sf = np.nan
                        i0 = np.nan
                        di = np.nan

                    cut_due_to_vn = (
                        vn_max_kv is not None
                        and float(vn.loc[hv]) > float(vn_max_kv) + 1e-9
                    )

                    cut_due_to_sens = (
                        np.isfinite(sf)
                        and np.isfinite(i0)
                        and i0 >= float(min_working_current_ka)
                        and sf < float(sensitivity_threshold)
                    )

                    cut_due_to_di_min = (
                        np.isfinite(di)
                        and abs(di) < float(di_min_ka)
                    )

                    if cut_due_to_vn or cut_due_to_sens or cut_due_to_di_min:
                        trafo_info = {
                            "el_type": "trafo3w",
                            "el_index": tidx,
                            "hv_bus": hv,
                            "mv_bus": mv,
                            "lv_bus": lv,
                            "boundary_bus_inside": int(u),  
                            "boundary_bus_outside": int(v),
                            "boundary_bus_other": int(z),
                            "reason": "vn_above_vn_max" if cut_due_to_vn else "up_below_current_sensitivity",
                            "from_vn_kv": vn_u,
                            "to_vn_kv": vn_v,
                            "sf_max": sf,
                            "i0_max_ka": i0,
                            "dI_max_ka": di,
                            "sensitivity_threshold": float(sensitivity_threshold),
                            "min_working_current_ka": float(min_working_current_ka),
                            "dI_min_ka": float(di_min_ka),
                        }
                        if is_upward:
                            upper_boundaries.append(trafo_info)
                        else:
                            lower_boundaries.append(trafo_info)

                        if keep_boundary_outside_bus:
                            kept_buses.add(int(v))
                            kept_buses.add(int(z))
                        continue

                if v not in visited:
                    visited.add(v)
                    visited.add(z)
                    kept_buses.add(v)
                    kept_buses.add(z)
                    queue.append(v)
                    queue.append(z)

                continue

            # ==========================================================
            # IMPEDANCE
            # ==========================================================
            if el_id[0] == "impedance":
                iidx = int(el_id[1])

                # local upward traversal?
                is_upward = vn_v > vn_u + 1e-9

                if is_upward or cut_downward_elements:

                    # sensitivity data
                    if iidx in impedance_sens_df.index:
                        sf = float(impedance_sens_df.at[iidx, "sf_max"]) if "sf_max" in impedance_sens_df.columns else np.nan
                        i0 = float(impedance_sens_df.at[iidx, "i0_max_ka"]) if "i0_max_ka" in impedance_sens_df.columns else np.nan
                        di = float(impedance_sens_df.at[iidx, "dI_max_ka"]) if "dI_max_ka" in impedance_sens_df.columns else np.nan
                    else:
                        sf = np.nan
                        i0 = np.nan
                        di = np.nan

                    cut_due_to_vn = (
                        vn_max_kv is not None
                        and vn_v > float(vn_max_kv) + 1e-9
                    )

                    cut_due_to_sens = (
                        np.isfinite(sf)
                        and np.isfinite(i0)
                        and i0 >= float(min_working_current_ka)
                        and sf < float(sensitivity_threshold)
                    )

                    cut_due_to_di_min = (
                        np.isfinite(di)
                        and abs(di) < float(di_min_ka)
                    )

                    if cut_due_to_vn or cut_due_to_sens or cut_due_to_di_min:
                        if is_upward:
                            hv_bus = v
                            lv_bus = u
                        else:
                            hv_bus = u
                            lv_bus = v

                        imp_info = {
                            "el_type": "impedance",
                            "el_index": iidx,
                            "hv_bus": hv_bus,
                            "lv_bus": lv_bus,
                            "boundary_bus_inside": int(u),
                            "boundary_bus_outside": int(v),
                            "reason": "vn_above_vn_max" if cut_due_to_vn else "up_below_current_sensitivity",
                            "from_vn_kv": vn_u,
                            "to_vn_kv": vn_v,
                            "sf_max": sf,
                            "i0_max_ka": i0,
                            "dI_max_ka": di,
                            "sensitivity_threshold": float(sensitivity_threshold),
                            "min_working_current_ka": float(min_working_current_ka),
                            "dI_min_ka": float(di_min_ka),
                        }
                        if is_upward:
                            upper_boundaries.append(imp_info)
                        else:
                            lower_boundaries.append(imp_info)

                        continue

                if v not in visited:
                    visited.add(v)
                    kept_buses.add(v)
                    queue.append(v)

    return kept_buses, upper_boundaries, lower_boundaries

Functions to create external grids at the upper boundaries (i.e., cuts of the grids towards higher voltage levels)

In [ ]:
def find_trafo_from_ext_grid(net, subnet, graph):
    """
    This function is used if no external grid exists in the reduced grid.
    It finds the transformer connected to the external grid in the original model.
    """

    el_pairs, el_adj = create_sets(net)
    
    queue = net.ext_grid["bus"].tolist()
    visited = set(queue)
    visited_element_dir = set()
    created = set()

    while queue:
        u = queue.pop(0)

        # --------------------------------------------------------------
        # 1) non-trafo neighbors
        # --------------------------------------------------------------
        for v in graph.neighbors(u):
            v = int(v)
            if frozenset((u, v)) in el_pairs:
                continue
            if v not in visited:
                visited.add(v)
                queue.append(v)

        # --------------------------------------------------------------
        # 2) neighbors
        # --------------------------------------------------------------
        for v, el_id in el_adj.get(u, []):
            v = int(v)
            if v > u:
                key = (el_id, int(u), int(v))
            else:
                key = (el_id, int(v), int(u))
            if key in visited_element_dir:
                continue
            visited_element_dir.add(key)

            # ==========================================================
            # 2W TRAFO
            # ==========================================================
            if el_id[0] == "trafo":
                tidx = int(el_id[1])

                subnet_boundary_trafo = subnet.trafo[subnet.trafo.index == tidx]
                if subnet_boundary_trafo.empty:
                    if v not in visited:
                        visited.add(v)
                        queue.append(v)
                else:
                    vm = float(net.res_bus.vm_pu.loc[u])
                    va = float(net.res_bus.va_degree.loc[u])
                    pp.create_ext_grid(subnet, bus=u, vm_pu=vm, va_degree=va)
                    created.add(u)

            # ==========================================================
            # 3W TRAFO
            # ==========================================================
            if el_id[0] == "trafo3w":
                tidx = int(el_id[1])

                subnet_boundary_trafo3w = subnet.trafo3w[subnet.trafo3w.index == tidx]
                if subnet_boundary_trafo3w.empty:
                    if v not in visited:
                        visited.add(v)
                        queue.append(v)
                else:
                    vm = float(net.res_bus.vm_pu.loc[u])
                    va = float(net.res_bus.va_degree.loc[u])
                    pp.create_ext_grid(subnet, bus=u, vm_pu=vm, va_degree=va)
                    created.add(u)

            # ==========================================================
            # IMPEDANCE
            # ==========================================================
            if el_id[0] == "impedance":
                iidx = int(el_id[1])

                subnet_boundary_impedance = subnet.impedance[subnet.impedance.index == iidx]
                if subnet_boundary_impedance.empty:
                    if v not in visited:
                        visited.add(v)
                        queue.append(v)
                else:
                    vm = float(net.res_bus.vm_pu.loc[u])
                    va = float(net.res_bus.va_degree.loc[u])
                    pp.create_ext_grid(subnet, bus=u, vm_pu=vm, va_degree=va)
                    created.add(u)

    return subnet, created


def add_boundary_ext_grids(subnet, net, boundaries, subgraph, graph):
    """
    Creates ext_grids at the boundary buses that remains inside the subnet.
    It considers the boundary buses met in upstream direction.
    """

    created = set()
    created_pq = set()
    boundary_list = set()

    for bnd in boundaries:
        
        btype = bnd["el_type"]

        if (btype == "trafo3w") or (btype == "trafo"):
            b = int(bnd["hv_bus"])
        else: 
            b = int(bnd["lv_bus"])

        boundary_list.add(b)
        vm = float(net.res_bus.vm_pu.loc[b])
        va = float(net.res_bus.va_degree.loc[b])

        if b not in created:
            pp.create_ext_grid(subnet, bus=b, vm_pu=vm, va_degree=va)
            created.add(b)

        if btype == "trafo3w":
            b_pq = bnd["boundary_bus_other"]         
            v = list(subgraph.neighbors(b_pq)) 

            if len(v) == 2: 
                idx = bnd["el_index"]
                if bnd["mv_bus"] == b_pq:
                    p = net.res_trafo3w.p_mv_mw.loc[idx]
                    q = net.res_trafo3w.q_mv_mvar.loc[idx]
                else:
                    p = net.res_trafo3w.p_lv_mw.loc[idx]
                    q = net.res_trafo3w.q_lv_mvar.loc[idx]

                pp.create_sgen(subnet, bus=b_pq, p_mw=p, q_mvar=q)
                created_pq.add(b_pq)

    if subnet.ext_grid.empty:
        subnet, b = find_trafo_from_ext_grid(net, subnet, graph)
        created = created.union(b)

    return created, created_pq

Functions to create power injections at the lower boundaries (i.e., cuts of the grids towards lower voltage levels)

In [ ]:
def compensate_pq_inj_for_elements_connected_to_bus(net, pq, b, str):
    """
    This function compensates for already existing loads, sgens, or other power injection elements
    already existing at the boundary bus.
    """

    def compensate_pq(net, b, str, element):
        res_el = "res_" + element

        if np.any(net[element][net[element].bus==b]):
            if str == "active":
                val = net[res_el].p_mw[net[element].bus==b].sum()
            elif str == "reactive":
                val = net[res_el].q_mvar[net[element].bus==b].sum()
        else:
            val = 0

        return val

    pq += compensate_pq(net, b, str, "load")        # compensation for connected loads
    pq -= compensate_pq(net, b, str, "sgen")        # compensation for connected sgens
    pq -= compensate_pq(net, b, str, "gen")         # compensation for connected gens
    pq += compensate_pq(net, b, str, "shunt")       # compensation for connected shunts
    pq += compensate_pq(net, b, str, "ward")        # compensation for connected ward
    pq += compensate_pq(net, b, str, "xward")       # compensation for connected xward

    return pq


def add_boundary_pq_injections(subnet, net, boundaries, subgraph):
    """
    Creates PQ injections at the boundary bus that remains inside the subnet.
    It considers the boundary buses met in downstream direction.
    """

    created = set()

    for bnd in boundaries:

        btype = bnd["el_type"]
        if btype == "trafo":
            b = int(bnd["lv_bus"])
            tr_idx = int(bnd["el_index"])

            v = list(subgraph.neighbors(b))
            create_pq = True
            if len(v)>1:
                b_volt = subnet.bus.vn_kv.loc[b]
                for it in v:
                    v_volt = subnet.bus.vn_kv.loc[it]
                    if v_volt == b_volt:
                        create_pq = False

            if create_pq:
                p = float(net.res_trafo.p_lv_mw.loc[tr_idx])
                q = float(net.res_trafo.q_lv_mvar.loc[tr_idx])

                if b not in created:
                    p = compensate_pq_inj_for_elements_connected_to_bus(net, p, b, "active")
                    q = compensate_pq_inj_for_elements_connected_to_bus(net, q, b, "reactive")

                pp.create_sgen(subnet, bus=b, p_mw=p, q_mvar=q)
                created.add(b)
            
        elif bnd["el_type"] == "trafo3w":
            b_mv = int(bnd["mv_bus"])
            b_lv = int(bnd["lv_bus"])
            tr_idx = int(bnd["el_index"])

            v_mv = list(subgraph.neighbors(b_mv))
            if len(v_mv)>2:
                continue

            v_lv = list(subgraph.neighbors(b_lv))
            if len(v_lv)>2:
                continue

            p_mv = float(net.res_trafo3w.p_mv_mw.loc[tr_idx])
            q_mv = float(net.res_trafo3w.q_mv_mvar.loc[tr_idx])
            p_lv = float(net.res_trafo3w.p_lv_mw.loc[tr_idx])
            q_lv = float(net.res_trafo3w.q_lv_mvar.loc[tr_idx])

            if b_mv not in created:
                p_mv = compensate_pq_inj_for_elements_connected_to_bus(net, p_mv, b_mv, "active")
                q_mv = compensate_pq_inj_for_elements_connected_to_bus(net, q_mv, b_mv, "reactive")

            if b_lv not in created:
                p_lv = compensate_pq_inj_for_elements_connected_to_bus(net, p_lv, b_lv, "active")
                q_lv = compensate_pq_inj_for_elements_connected_to_bus(net, q_lv, b_lv, "reactive")
            
            pp.create_sgen(subnet, bus=b_mv, p_mw=p_mv, q_mvar=q_mv)
            pp.create_sgen(subnet, bus=b_lv, p_mw=p_lv, q_mvar=q_lv)
            created.add(b_mv)
            created.add(b_lv)

        else:
            b = int(bnd["hv_bus"])
            imp_idx = int(bnd["el_index"])

            b_lv = int(bnd["lv_bus"])
            if np.any(subnet.bus.index==b_lv):
                continue
            
            if net.impedance.from_bus.loc[imp_idx] == b:
                p = - float(net.res_impedance.p_from_mw.loc[imp_idx])
                q = - float(net.res_impedance.q_from_mvar.loc[imp_idx])
            else:
                p = - float(net.res_impedance.p_to_mw.loc[imp_idx])
                q = - float(net.res_impedance.q_to_mvar.loc[imp_idx])

            pp.create_sgen(subnet, bus=b, p_mw=p, q_mvar=q)
            created.add(b)

    return created

Main function calling all the sub-functions for grid reduction

In [ ]:
def build_reduced_network(net, start_bus, sensitivity_threshold=0.05, 
                        min_working_current_ka=0.01, vn_max_kv=None, 
                        deltap_mw=1.0, deltaq_mvar=0.0, 
                        cut_downward_elements=True):
    """
    Complete workflow based on classical bus sensitivity calculation:
      1) assumes base PF already exists in net
      2) runs power flows with pwr inj variation
      3) computes element sensitivities
      4) cuts elements by sensitivity / vn_max / i_min
      5) builds subnet
      6) adds boundary ext_grids or power injections

    Returns
    -------
    subnet, trafo_sens_df, trafo3w_sens_df, impedance_sens_df, kept_buses, boundaries, created_ext_grids, created_pq_injections
    """

    if ~np.any(net.bus.index == start_bus):
        print("The selected bus was not found in the considered grid")
        return net, np.empty, np.empty(0), np.empty(0), np.empty(0), np.empty(0), np.empty(0)

    net_post = copy.deepcopy(net)
    pp.runpp(net, run_control=False, max_iteration=100)

    pp.create_sgen(net_post, bus=start_bus, p_mw=deltap_mw, q_mvar=deltaq_mvar)
    pp.runpp(net_post, run_control=False, max_iteration=100)

    trafo_sens_df = calc_trafo_current_sensitivity_from_power_flow(
        net,
        net_post,
        min_i_ka=1e-6)
    
    trafo3w_sens_df = calc_trafo3w_current_sensitivity_from_power_flow(
        net,
        net_post,
        min_i_ka=1e-6)
    
    impedance_sens_df = calc_impedance_current_sensitivity_from_power_flow(
        net,
        net_post,
        min_i_ka=1e-6)
    
    graph = top.create_nxgraph(net, respect_switches=True)

    kept_buses, hv_boundaries, lv_boundaries = cut_by_sensitivity(
        net, graph,
        start_bus=start_bus,
        trafo_sens_df=trafo_sens_df,
        trafo3w_sens_df=trafo3w_sens_df,
        impedance_sens_df=impedance_sens_df,
        sensitivity_threshold=sensitivity_threshold,
        min_working_current_ka=min_working_current_ka,
        di_min_ka=1e-4,
        vn_max_kv=vn_max_kv,
        cut_downward_elements=cut_downward_elements,
        keep_boundary_outside_bus=True)

    subnet = select_subnet(net, buses=list(kept_buses), include_results=True)
    subnet.user_pf_options = net.user_pf_options
    subgraph = top.create_nxgraph(subnet, respect_switches=True)

    created_ext_grids, created_pq = add_boundary_ext_grids(subnet, net, hv_boundaries, subgraph, graph)
    created_pq_injections = add_boundary_pq_injections(subnet, net, lv_boundaries, subgraph)
    created_pq_injections = created_pq_injections.union(created_pq)

    boundaries = {}
    boundaries["hv"] = hv_boundaries
    boundaries["lv"] = lv_boundaries

    try:
        trafo_sens_sorted = trafo_sens_df.sort_values(by="sf_max", ascending=False)
    except:
        trafo_sens_sorted = None
    try:
        trafo3w_sens_sorted = trafo3w_sens_df.sort_values(by="sf_max", ascending=False)
    except: 
        trafo3w_sens_sorted = None
    try:
        impedance_sens_sorted = impedance_sens_df.sort_values(by="sf_max", ascending=False)
    except:
        impedance_sens_sorted = None

    return subnet, trafo_sens_sorted, trafo3w_sens_sorted, impedance_sens_sorted, boundaries, created_ext_grids, created_pq_injections

#### Example 1 - Reduction based only on sensitivity of upstream components

In this first example, it will be shown how the grid is reduced based on the sensitivity factors. 
No cuts due to voltage levels will be applied. 
Cuts will be applied only in upstream direction.

In [ ]:
start_bus = 4120    # Select the bus of interest around which you want to reduce the grid

# Call the main function for grid reduction
subnet, trafo_sens_df, trafo3w_sens_df, impedance_sens_df, boundaries, created_ext_grids, created_pq_injections = build_reduced_network(
    net, 
    start_bus=start_bus,            # start bus considered for the reduction
    sensitivity_threshold=0.05,     # threshold to decide if cutting or not the subnet
    min_working_current_ka=0.001,   # minimum current limit considered for the cutting
    vn_max_kv=1E6,                  # maximum voltage limit considered for the cutting
    deltap_mw=1.0,                  # delta of active power toapplied for the sensitivity calculation
    deltaq_mvar=0.0,                # delta of reactive power toapplied for the sensitivity calculation
    cut_downward_elements=False)     # decide if apply cuts also in downstream direction (lower voltage levels) or not

Display the details about the starting bus:

In [ ]:
if ~np.any(net.bus.index == start_bus):
    print("The selected bus was not found in the considered grid")
else:
    display(net.bus.loc[start_bus])


You can see the comparison between original and reduced grid:

In [ ]:
# Original grid
display(net)

In [ ]:
# Reduced grid
display(subnet)

You can print a summary of the reduction process:

In [ ]:
print("kept buses:", len(subnet.bus))
try:
    print("boundaries:", len(boundaries["hv"])+len(boundaries["lv"]))
except: 
    print("boundaries:", len(boundaries))
print("ext_grids created:", len(created_ext_grids))
print("ext_grid buses:", subnet.ext_grid.bus.tolist())
print("pq_injections_created:", len(created_pq_injections))
print("pq_injection buses:", subnet.sgen[-len(created_pq_injections):].bus.tolist())

You can dentify which trasformers have a sensitivity larger than the sensitivity threshold (5% --> 0.05 p.u.). 
No cut is applied at these transformers, as those transformers and the grid above is strongly connected to and affected by changes at the bus of interest.

In [ ]:
try:
    display(trafo_sens_df[trafo_sens_df["sf_max"].values > 0.05])
except:
    display("No transformer has been found in this grid")

In [ ]:
try:
    display(trafo3w_sens_df[trafo3w_sens_df["sf_max"].values > 0.05])
except:
    display("No transformer has been found in this grid")
    

You can compare the results of the power flow in the original grid and in the reduced one:

In [ ]:
pp.runpp(subnet, run_control=False, max_iteration=100)
pp.runpp(net, run_control=False, max_iteration=100)
kept_buses = subnet.bus.index
vm_full_net = net.res_bus.loc[kept_buses,"vm_pu"].values
vm_subnet = subnet.res_bus["vm_pu"].values
diff_vm = vm_full_net - vm_subnet
max_diff = max(abs(diff_vm))

display("Maximum voltage magnitude difference between original and reduced grid (per unit): " + str(max_diff))

#### Example 2 - Reduction based only on sensitivity of both upstream and downstream components

In this example, the reduced grid will also consider cuts towards the downstream direction. 
No cuts due to maximum voltage levels will be applied. 

In [ ]:
# Call the main function for grid reduction  
# ---> set cut_downstream elements = True for cutting also in the downstream direction
subnet, trafo_sens_df, trafo3w_sens_df, impedance_sens_df, boundaries, created_ext_grids, created_pq_injections = build_reduced_network(
    net, 
    start_bus=start_bus,            # start bus considered for the reduction
    sensitivity_threshold=0.05,     # threshold to decide if cutting or not the subnet
    min_working_current_ka=0.001,   # minimum current limit considered for the cutting
    vn_max_kv=1E6,                  # maximum voltage limit considered for the cutting
    deltap_mw=1.0,                  # delta of active power toapplied for the sensitivity calculation
    deltaq_mvar=0.0,                # delta of reactive power toapplied for the sensitivity calculation
    cut_downward_elements=True)     # decide if apply cuts also in downstream direction (lower voltage levels) or not

You can see the summary of the reduction process. The cuts in the downstream direction will lead to remove several low voltage parts of the grid and to replace them with equivalent power injections.

In [ ]:
print("kept buses:", len(subnet.bus))
try:
    print("boundaries:", len(boundaries["hv"])+len(boundaries["lv"]))
except: 
    print("boundaries:", len(boundaries))
print("ext_grids created:", len(created_ext_grids))
print("ext_grid buses:", subnet.ext_grid.bus.tolist())
print("pq_injections_created:", len(created_pq_injections))
print("pq_injection buses:", subnet.sgen[-len(created_pq_injections):].bus.tolist())

You can get an overall view of the elements present in the created subnet:

In [ ]:
# Print the elements of the reduced grid
display(subnet)

You can compare again the results of the power flow for the original and reduced grid:

In [ ]:
pp.runpp(subnet, run_control=False, max_iteration=100)
pp.runpp(net, run_control=False, max_iteration=100)
kept_buses = subnet.bus.index
vm_full_net = net.res_bus.loc[kept_buses,"vm_pu"].values
vm_subnet = subnet.res_bus["vm_pu"].values
diff_vm = vm_full_net - vm_subnet
max_diff = max(abs(diff_vm))

display("Maximum voltage magnitude difference between original and reduced grid (per unit): " + str(max_diff))

#### Example 3 - Reduction based on sensitivity (upstream and downstream) and on maximum voltage level

In this example, the reduced grid will only consider parts of the grid up to a user defined voltage level. For parts of the grid below the maximum voltage, the reduction will be still executed based on the sensitivity factors.

In [ ]:
# Call the main function for grid reduction  
# ---> set vn_max_kv to the desired max voltage level
subnet, trafo_sens_df, trafo3w_sens_df, impedance_sens_df, boundaries, created_ext_grids, created_pq_injections = build_reduced_network(
    net, 
    start_bus=start_bus,            # start bus considered for the reduction
    sensitivity_threshold=0.05,     # threshold to decide if cutting or not the subnet
    min_working_current_ka=0.001,   # minimum current limit considered for the cutting
    vn_max_kv=50,                   # maximum voltage limit considered for the cutting
    deltap_mw=1.0,                  # delta of active power toapplied for the sensitivity calculation
    deltaq_mvar=0.0,                # delta of reactive power toapplied for the sensitivity calculation
    cut_downward_elements=True)     # decide if apply cuts also in downstream direction (lower voltage levels) or not

Here is the summary of the reduction process:

In [ ]:
print("kept buses:", len(subnet.bus))
try:
    print("boundaries:", len(boundaries["hv"])+len(boundaries["lv"]))
except: 
    print("boundaries:", len(boundaries))
print("ext_grids created:", len(created_ext_grids))
print("ext_grid buses:", subnet.ext_grid.bus.tolist())
print("pq_injections_created:", len(created_pq_injections))
print("pq_injection buses:", subnet.sgen[-len(created_pq_injections):].bus.tolist())

You can get an overall view of the elements present in the created subnet:

In [ ]:
# Print the elements of the reduced grid
display(subnet)

You check the maximum voltage existing in the reduced grid and verify that the buses above the threshold belong to transformers where the cut was applied.

In [ ]:
# Print the buses sorting them by the highest voltage
display(subnet.bus.sort_values(by="vn_kv", ascending=False))

In [ ]:
# Print the transformers sorting them by the highest voltage
display(subnet.trafo.sort_values(by="vn_hv_kv", ascending =False))

You can compare again the results of the power flow for the original and reduced grid:

In [ ]:
pp.runpp(subnet, run_control=False, max_iteration=100)
pp.runpp(net, run_control=False, max_iteration=100)
kept_buses = subnet.bus.index
vm_full_net = net.res_bus.loc[kept_buses,"vm_pu"].values
vm_subnet = subnet.res_bus["vm_pu"].values
diff_vm = vm_full_net - vm_subnet
max_diff = max(abs(diff_vm))

display("Maximum voltage magnitude difference between original and reduced grid (per unit): " + str(max_diff))